# Final Kimble 4 mL tube geometry test

This notebook uses a **300 uL filtered tip** from `A1` of the rack in tip-carrier site `1` to verify the calibrated rim, half-depth, and inner bottom of a `Kimble_tube_4mL_Rb` in the frontmost site (site `0`) of a Hamilton 32-position tube carrier.

The confirmed tube seating offset is **15.0 mm**. The confirmed inner-bottom target is **5.0 mm above the modeled outside bottom**. Keep the deck clear and supervise every move.

**Run order:** Execute cells from top to bottom once. After hardware setup, do not rerun the deck-layout cell because it creates a new, uninitialized `LiquidHandler`. Do not run the stop cell until testing or recovery is complete.

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import (
    Kimble_tube_4mL_Rb,
    hamilton_96_tiprack_1000uL_filter,
    hamilton_96_tiprack_300uL_filter,
)
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL,
)

In [ ]:
# Final calibrated constants. PyLabRobot channel indices are zero-based.
CHANNEL = 0  # physical channel 1 (backmost channel)
TUBE_SITE = 0  # frontmost carrier position
TIP_CARRIER_SITE = 1  # 300 uL filtered-tip rack; PyLabRobot sites are zero-based
TIP_POSITION = "A1"
KIMBLE_SEATING_Z_OFFSET_MM = 15.0
INNER_BOTTOM_ABOVE_OUTSIDE_MM = 5.0
RIM_CLEARANCE_MM = 5.0
SAFE_TRAVERSE_Z = 247.6
RIM_HOLD_SECONDS = 10.0

backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_rack_1000ul = hamilton_96_tiprack_1000uL_filter(name="tips_1000ul_filter")
tip_rack_300ul = hamilton_96_tiprack_300uL_filter(name="tips_300ul_filter")
tip_carrier[0].assign_child_resource(tip_rack_1000ul)
tip_carrier[TIP_CARRIER_SITE].assign_child_resource(tip_rack_300ul)
lh.deck.assign_child_resource(tip_carrier, rails=25)

tube_carrier = hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL(
    name="tube_carrier"
)
kimble_tube = Kimble_tube_4mL_Rb(name="kimble_4mL_site_0")

# Center the 10.0 mm tube in the 10.8 mm holder and apply the confirmed
# 15.0 mm Z seating offset for this tube/insert combination.
tube_location_in_holder = Coordinate(
    (tube_carrier[TUBE_SITE].get_size_x() - kimble_tube.get_size_x()) / 2,
    (tube_carrier[TUBE_SITE].get_size_y() - kimble_tube.get_size_y()) / 2,
    KIMBLE_SEATING_Z_OFFSET_MM,
)
tube_carrier[TUBE_SITE].assign_child_resource(
    kimble_tube, location=tube_location_in_holder
)
lh.deck.assign_child_resource(tube_carrier, rails=19)

print(lh.deck.summary())

In [ ]:
# Derive and verify every final target before enabling motion.
assert kimble_tube.get_size_x() == 10
assert kimble_tube.get_size_y() == 10
assert kimble_tube.get_size_z() == 75
assert TUBE_SITE == min(tube_carrier.sites), "Site 0 must be the frontmost site"
assert TIP_CARRIER_SITE == 1, "300 uL filtered-tip rack must be in tip-carrier site 1"
assert tip_rack_300ul[TIP_POSITION][0].has_tip(), f"No tip modeled at {TIP_POSITION}"
selected_tip = tip_rack_300ul[TIP_POSITION][0].get_tip()
assert selected_tip.nominal_volume == 300
assert selected_tip.has_filter

tube_outside_bottom = kimble_tube.get_absolute_location(x="c", y="c", z="b")
tube_rim = kimble_tube.get_absolute_location(x="c", y="c", z="t")
holder_center = tube_carrier[TUBE_SITE].get_absolute_location(x="c", y="c", z="b")
tip_rack_top = tip_rack_300ul.get_absolute_location(z="t").z

CENTER_X = holder_center.x
CENTER_Y = holder_center.y
MODELED_OUTSIDE_BOTTOM_Z = tube_outside_bottom.z
CALIBRATED_INNER_BOTTOM_Z = MODELED_OUTSIDE_BOTTOM_Z + INNER_BOTTOM_ABOVE_OUTSIDE_MM
CALIBRATED_RIM_Z = tube_rim.z
CALIBRATED_INNER_DEPTH_MM = CALIBRATED_RIM_Z - CALIBRATED_INNER_BOTTOM_Z
HALF_DEPTH_Z = CALIBRATED_INNER_BOTTOM_Z + CALIBRATED_INNER_DEPTH_MM / 2
ABOVE_RIM_Z = CALIBRATED_RIM_Z + RIM_CLEARANCE_MM

assert abs(tube_outside_bottom.x - CENTER_X) < 1e-6
assert abs(tube_outside_bottom.y - CENTER_Y) < 1e-6
assert abs(CENTER_X - 519.50) < 0.01
assert abs(CENTER_Y - 77.50) < 0.01
assert abs(MODELED_OUTSIDE_BOTTOM_Z - 140.88) < 0.01
assert abs(CALIBRATED_INNER_BOTTOM_Z - 145.88) < 0.01
assert abs(HALF_DEPTH_Z - 180.88) < 0.01
assert abs(CALIBRATED_RIM_Z - 215.88) < 0.01
assert abs(ABOVE_RIM_Z - 220.88) < 0.01
assert abs(CALIBRATED_INNER_DEPTH_MM - 70.0) < 0.01
assert SAFE_TRAVERSE_Z > tip_rack_top
assert ABOVE_RIM_Z < SAFE_TRAVERSE_Z
assert CALIBRATED_INNER_BOTTOM_Z < HALF_DEPTH_Z < CALIBRATED_RIM_Z

print("=== FINAL KIMBLE 4 mL GEOMETRY (absolute deck coordinates, mm) ===")
print(f"Tube/hole center:          X={CENTER_X:.2f}, Y={CENTER_Y:.2f}")
print(f"Modeled outside bottom:    Z={MODELED_OUTSIDE_BOTTOM_Z:.2f}  (not commanded)")
print(f"CALIBRATED INNER BOTTOM:   Z={CALIBRATED_INNER_BOTTOM_Z:.2f}")
print(f"CALIBRATED HALF-DEPTH:     Z={HALF_DEPTH_Z:.2f}")
print(f"CALIBRATED RIM:            Z={CALIBRATED_RIM_Z:.2f}")
print(f"ABOVE-RIM TARGET:          Z={ABOVE_RIM_Z:.2f}")
print(f"Calibrated inner depth:    {CALIBRATED_INNER_DEPTH_MM:.2f} mm")
print(f"300 uL tip source:         carrier site {TIP_CARRIER_SITE}, {TIP_POSITION}")
print(f"300 uL tip-rack top:       Z={tip_rack_top:.2f}")
print(f"SAFE TRAVERSE:             Z={SAFE_TRAVERSE_Z:.2f}")
print(f"Traverse above tip rack:   {SAFE_TRAVERSE_Z - tip_rack_top:.2f} mm")
print(f"Tube location in holder:   dX={tube_location_in_holder.x:.2f}, "
      f"dY={tube_location_in_holder.y:.2f}, dZ={tube_location_in_holder.z:.2f}")

## Connect and run

Confirm that the physical deck matches the printed layout before setup. Tip-carrier site 1 must contain the 300 uL filtered-tip rack with a tip at A1. Tube-carrier site 0 must contain the Kimble tube; no 1.5 mL tube should be present there. After setup succeeds, do not rerun the deck-layout cell.

In [ ]:
if lh.setup_finished:
    print("Liquid handler setup is already complete; setup was not repeated.")
else:
    await lh.setup(skip_autoload=True)

In [ ]:
async def confirm(message: str):
    response = input(f"\n{message} Type GO and press Enter to continue: " ).strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the geometry test.")


async def run_kimble_geometry_test():
    tip_is_attached = False
    test_completed = False

    await confirm(
        f"Verify: 300 uL filtered tip at tip-carrier site {TIP_CARRIER_SITE}, "
        f"{TIP_POSITION}; Kimble 4 mL tube at frontmost tube site 0; deck clear."
    )
    print(
        f"Picking up 300 uL filtered tip at carrier site {TIP_CARRIER_SITE}, "
        f"{TIP_POSITION}, with physical channel 1 (PyLabRobot index 0)."
    )
    # await lh.pick_up_tips(tip_rack_300ul[TIP_POSITION], use_channels=[CHANNEL])
    # tip_is_attached = True
    await lh.pick_up_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
    tip_is_attached = True

    try:
        await lh.prepare_for_manual_channel_operation(CHANNEL)

        print(f"Raising tip end to SAFE TRAVERSE Z={SAFE_TRAVERSE_Z:.2f}.")
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)

        await confirm(
            f"Ready to move at safe Z to tube center X={CENTER_X:.2f}, Y={CENTER_Y:.2f}."
        )
        await lh.move_channel_x(CHANNEL, CENTER_X)
        await lh.move_channel_y(CHANNEL, CENTER_Y)

        await confirm(
            f"Ready to descend to ABOVE-RIM Z={ABOVE_RIM_Z:.2f} "
            f"({RIM_CLEARANCE_MM:.1f} mm above calibrated rim Z={CALIBRATED_RIM_Z:.2f})."
        )
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=ABOVE_RIM_Z)
        print(f"Holding above the rim for {RIM_HOLD_SECONDS:.0f} seconds.")
        await asyncio.sleep(RIM_HOLD_SECONDS)

        await confirm(f"Ready to descend to CALIBRATED HALF-DEPTH Z={HALF_DEPTH_Z:.2f}.")
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=HALF_DEPTH_Z)

        await confirm(
            f"Ready to descend to the confirmed CALIBRATED INNER BOTTOM "
            f"Z={CALIBRATED_INNER_BOTTOM_Z:.2f}."
        )
        await lh.backend.move_channel_tool_z(
            channel_idx=CHANNEL, z=CALIBRATED_INNER_BOTTOM_Z
        )
        print("Calibrated inner-bottom target reached.")
        await confirm(
            f"Inspection complete; ready to raise and return the tip to {TIP_POSITION}."
        )

        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
        # await lh.drop_tips(tip_rack_300ul[TIP_POSITION], use_channels=[CHANNEL])
        await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
        tip_is_attached = False
        test_completed = True
        print(f"Final geometry test complete; 300 uL tip returned to {TIP_POSITION}.")
    finally:
        if tip_is_attached:
            print("Test interrupted with a tip attached. Attempting a safe vertical raise.")
            try:
                await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
                print("Channel raised. Use the recovery cell below to return the tip.")
            except Exception as recovery_error:
                print(f"Automatic raise failed: {recovery_error!r}")
                print("Do not move the deck or tube until the channel is recovered manually.")

    return test_completed

In [16]:
test_completed = await run_kimble_geometry_test()

Picking up 300 uL filtered tip at carrier site 1, A1, with physical channel 1 (PyLabRobot index 0).
Raising tip end to SAFE TRAVERSE Z=247.60.
Holding above the rim for 10 seconds.
Calibrated inner-bottom target reached.
Test interrupted with a tip attached. Attempting a safe vertical raise.
Channel raised. Use the recovery cell below to return the tip.


ChannelizedError: ChannelizedError(errors={0: HardwareError('Z-drive movement error')}, raw_response=C0TRid0090er99/00 P102/62kz370 000 000 000 000 000 000 000vz000 000 000 000 000 000 000 000)

## Recovery cell

Run this only if the test was interrupted after the 300 uL filtered tip at A1 in tip-carrier site 1 was picked up and the channel still has that tip. The test routine already attempts the safe vertical raise; this cell repeats that raise before returning the tip.

In [17]:
# await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
# await lh.drop_tips(tip_rack_300ul[TIP_POSITION], use_channels=[CHANNEL])
await lh.drop_tips(tip_rack_1000ul["A1"], use_channels=[0])
# print(f"Recovery complete; 300 uL tip returned to {TIP_POSITION}.")

## Stop

Run only after the final test or recovery is complete.

In [ ]:
# await lh.stop()

2026-07-04 10:10:11,563 - pylabrobot.io.usb - WARNING - Closing connection to USB device.
